# ECE405 Assignment 2 — Data Filtering for Language Modeling

Based on CS336 Assignment 4 (Stanford, Spring 2025)

## 2.1 Looking at the data

### Setup: Download sample WARC and WET files

In [ ]:
import os, shutil

# --- Configuration ---
REPO_URL = "https://github.com/bushuyeu/LLM-from-scratch.git"
REPO_DIR = "/content/LLM-from-scratch"
ASSIGNMENT_DIR = os.path.join(REPO_DIR, "ece405_assignment2")
BRANCH = "main"

# Clone / update the repo on the Colab kernel
if os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    # Clean up any leftover non-git directory
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

# Install dependencies:
# 1. Common pip packages needed by the notebook and cs336-data
!pip install -q warcio resiliparse fasttext tldextract xopen "numpy<2.0" tqdm nltk mmh3

# 2. cs336-basics (local sub-package required by cs336-data)
!pip install -e {ASSIGNMENT_DIR}/cs336-basics

# 3. cs336-data itself (--no-deps because pip can't resolve the uv-specific
#    local-path source for cs336-basics; we already installed it above)
!pip install --no-deps -e {ASSIGNMENT_DIR}

# Verify install
!pip show cs336-data

# Set up data directory
DATA_DIR = os.path.join(ASSIGNMENT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
WARC_URL = "https://data.commoncrawl.org/crawl-data/CC-MAIN-2025-18/segments/1744889135610.12/warc/CC-MAIN-20250417135010-20250417165010-00065.warc.gz"
WET_URL = "https://data.commoncrawl.org/crawl-data/CC-MAIN-2025-18/segments/1744889135610.12/wet/CC-MAIN-20250417135010-20250417165010-00065.warc.wet.gz"

WARC_PATH = os.path.join(DATA_DIR, "example.warc.gz")
WET_PATH = os.path.join(DATA_DIR, "example.warc.wet.gz")

In [3]:
# Download WARC file (approx 1GB compressed)
if not os.path.exists(WARC_PATH):
    !wget -O {WARC_PATH} "{WARC_URL}"
else:
    print(f"WARC file already exists at {WARC_PATH}")

WARC file already exists at /content/LLM-from-scratch/ece405_assignment2/data/example.warc.gz


In [4]:
# Download WET file
if not os.path.exists(WET_PATH):
    !wget -O {WET_PATH} "{WET_URL}"
else:
    print(f"WET file already exists at {WET_PATH}")

WET file already exists at /content/LLM-from-scratch/ece405_assignment2/data/example.warc.wet.gz


### (a) Examine the first page in the WARC file

In [5]:
from warcio.archiveiterator import ArchiveIterator

with open(WARC_PATH, "rb") as f:
    for i, record in enumerate(ArchiveIterator(f)):
        print(f"Record {i}: type={record.rec_type}")
        if record.rec_type == "response":
            print(f"  URL: {record.rec_headers.get_header('WARC-Target-URI')}")
            print(f"  Date: {record.rec_headers.get_header('WARC-Date')}")
            print(f"  Content-Type: {record.http_headers.get_header('Content-Type') if record.http_headers else 'N/A'}")
            print()
            # Print first 3000 chars of HTML content
            content = record.content_stream().read()
            print("--- First 3000 chars of raw content ---")
            print(content[:3000].decode('utf-8', errors='replace'))
            break

Record 0: type=warcinfo
Record 1: type=request
Record 2: type=response
  URL: http://0371rykj.com/ipfhsb/34.html
  Date: 2025-04-17T14:56:33Z
  Content-Type: text/html

--- First 3000 chars of raw content ---
<!DOCTYPE html>
<html>
 <head> 
   <meta charset="UTF-8">
  
  <title>&#x4EBA;&#x59BB;&#x2C;&#x56FD;&#x5185;&#x8001;&#x719F;&#x5987;&#x5BF9;&#x767D;&#x48;&#x44;&#x58;&#x58;&#x58;&#x58;&#x2C;&#x4E9A;&#x6D32;&#x41;&#x56;&#x65E0;&#x7801;&#x4E00;&#x533A;&#x4E1C;&#x4EAC;&#x70ED;&#x4E45;&#x4E45;&#x0D;</title>
  <meta name="keywords" content="&#x4EBA;&#x59BB;&#x2C;&#x56FD;&#x5185;&#x8001;&#x719F;&#x5987;&#x5BF9;&#x767D;&#x48;&#x44;&#x58;&#x58;&#x58;&#x58;&#x2C;&#x4E9A;&#x6D32;&#x41;&#x56;&#x65E0;&#x7801;&#x4E00;&#x533A;&#x4E1C;&#x4EAC;&#x70ED;&#x4E45;&#x4E45;&#x0D;" />
  <meta name="description" content="&#x4EBA;&#x59BB;&#x2C;&#x56FD;&#x5185;&#x8001;&#x719F;&#x5987;&#x5BF9;&#x767D;&#x48;&#x44;&#x58;&#x58;&#x58;&#x58;&#x2C;&#x4E9A;&#x6D32;&#x41;&#x56;&#x65E0;&#x7801;&#x4E00;&#x533A;&#x4E1

### (b) Examine the corresponding WET file

In [6]:
with open(WET_PATH, "rb") as f:
    for i, record in enumerate(ArchiveIterator(f)):
        if record.rec_type == "conversion":
            url = record.rec_headers.get_header('WARC-Target-URI')
            content = record.content_stream().read().decode('utf-8', errors='replace')
            print(f"URL: {url}")
            print(f"Content length: {len(content)} chars")
            print()
            print("--- Extracted text (first 3000 chars) ---")
            print(content[:3000])
            break

URL: http://0371rykj.com/ipfhsb/34.html
Content length: 3496 chars

--- Extracted text (first 3000 chars) ---
人妻,国内老熟妇对白HDXXXX,亚洲AV无码一区东京热久久
久久久久女人精品毛片,99久久精品无码一区二区毛片,被老外的又粗又大日出了水,一边吃奶一边哭乱抻又乱扭
恒溫恒濕試驗(yàn)箱
在線(xiàn)咨詢(xún)
上海林頻儀器股份有限公司Shanghai Linpin Instrument Stock Co Ltd
服務(wù)熱線(xiàn)：4000 662 888
手機(jī)咨詢(xún)：13818467052
首頁(yè)
林頻產(chǎn)品
試驗(yàn)箱系列
老化箱系列
非標(biāo)定制系列
ip防護(hù)系列
振動(dòng)跌落系列
成功案例
新聞中心
林頻新聞
行業(yè)新聞
常見(jiàn)問(wèn)題
解決方案
關(guān)于林頻
服務(wù)支持
聯(lián)系我們
您所在的位置：
恒溫恒濕試驗(yàn)箱 > 林頻產(chǎn)品 > ip防護(hù)系列 >
產(chǎn)品詳情/ products details
恒溫恒濕試驗(yàn)箱
產(chǎn)品用途
恒溫恒濕試驗(yàn)箱是航空、汽車(chē)、家電、科研等領(lǐng)域必備的測(cè)試設(shè)備，用于測(cè)試和確定電工、電子及其他產(chǎn)品及材料進(jìn)行高溫、低溫、濕熱度或恒定試驗(yàn)的溫度環(huán)境變化后的參數(shù)及性能。...
了解詳情 立即咨詢(xún)
產(chǎn)品參數(shù) / parameter
設(shè)備型號(hào) 工作室尺寸(D*W*H)mm 外型尺寸(D*W*H)mm
LRHS-101-LH 450×450×500 1160×1000×1610
LRHS-225-LH 500×600×750 1210×1150×1870
LRHS-504-LH 700×800×900 1260×1340×2070
LRHS-800-LH 800×1000×1000 1370×1550×2170
LRHS-1000-LH 1000×1000×1000 1560×155

### (d) Annotate 25 WET records

In [7]:
with open(WET_PATH, "rb") as f:
    count = 0
    for record in ArchiveIterator(f):
        if record.rec_type == "conversion":
            url = record.rec_headers.get_header('WARC-Target-URI')
            content = record.content_stream().read().decode('utf-8', errors='replace')
            print(f"\n{'='*80}")
            print(f"Record {count + 1}")
            print(f"URL: {url}")
            print(f"Content length: {len(content)} chars")
            print(f"First 500 chars:")
            print(content[:500])
            print(f"{'='*80}")
            count += 1
            if count >= 25:
                break


Record 1
URL: http://0371rykj.com/ipfhsb/34.html
Content length: 3496 chars
First 500 chars:
人妻,国内老熟妇对白HDXXXX,亚洲AV无码一区东京热久久
久久久久女人精品毛片,99久久精品无码一区二区毛片,被老外的又粗又大日出了水,一边吃奶一边哭乱抻又乱扭
恒溫恒濕試驗(yàn)箱
在線(xiàn)咨詢(xún)
上海林頻儀器股份有限公司Shanghai Linpin Instrument Stock Co Ltd
服務(wù)熱線(xiàn)：4000 662 888
手機(jī)咨詢(xún)：13818467052
首頁(yè)
林頻產(chǎn)品
試驗(yàn)箱系列
老化箱系列
非標(biāo)定制系列
ip防護(hù)系列
振動(dòng)跌落系列
成功案例
新聞中心
林頻新聞
行業(yè)新聞
常見(jiàn)問(wèn)題
解決方案
關(guān)于林頻
服務(wù)支持
聯(lián)系我們
您所在的位置：
恒溫恒濕試驗(yàn)箱 > 林頻產(chǎn)品 > ip防護(hù)系列 >
產(chǎn)品詳情/ products details
恒溫恒濕試驗(yàn)箱
產(chǎn)品用途
恒溫恒濕試驗(yàn)箱是航空、汽車(chē)、家電、科研等領(

Record 2
URL: http://10www.chinatikfans.com/home.php?mod=space&uid=4693&do=blog&classid=104&view=me
Content length: 2066 chars
First 500 chars:
lily_zl的日志 - &#1769;杰西达邦中国影迷会&#1769; - Powered by Discuz!
设为首页收藏本站
开启辅助访问 切换到窄版
帐号 自动登录 找回密码
密码
登录
加入我们
只需一步，快速开始
快捷导航
广场BBS
家园Space
每日签到
排行榜Ranklist
手机论坛
搜索
搜索
用户
&#1769;杰西达邦中国影迷会&#1769; › 日志
际遇
发布 日志
上传 相册
添加 分享
记录
日志
好友的日志
我的日志
随便看看
发表新日志
discussion on plot

## 2.2 HTML to text conversion

### (a) Extract text from HTML bytes using Resiliparse

In [8]:
from cs336_data.extract import extract_text_from_html_bytes

### (b) Compare Resiliparse extraction vs WET extraction on the first page

In [9]:
# Extract text from the first WARC response using our function, and compare with WET
from warcio.archiveiterator import ArchiveIterator

# Get Resiliparse extraction from WARC
resiliparse_text = None
with open(WARC_PATH, "rb") as f:
    for record in ArchiveIterator(f):
        if record.rec_type == "response":
            html_bytes = record.content_stream().read()
            resiliparse_text = extract_text_from_html_bytes(html_bytes)
            warc_url = record.rec_headers.get_header('WARC-Target-URI')
            break

# Get WET extraction
wet_text = None
with open(WET_PATH, "rb") as f:
    for record in ArchiveIterator(f):
        if record.rec_type == "conversion":
            wet_text = record.content_stream().read().decode('utf-8', errors='replace')
            break

print(f"URL: {warc_url}")
print(f"Resiliparse output length: {len(resiliparse_text)} chars")
print(f"WET output length:         {len(wet_text)} chars")
print()
print("=" * 80)
print("RESILIPARSE EXTRACTION (first 2000 chars)")
print("=" * 80)
print(resiliparse_text[:2000])
print()
print("=" * 80)
print("WET EXTRACTION (first 2000 chars)")
print("=" * 80)
print(wet_text[:2000])

URL: http://0371rykj.com/ipfhsb/34.html
Resiliparse output length: 10165 chars
WET output length:         3496 chars

RESILIPARSE EXTRACTION (first 2000 chars)
久久久久女人精品毛片,99久久精品无码一区二区毛片,被老外的又粗又大日出了水,一边吃奶一边哭乱抻又乱扭

        • <th id="gckmo"></th>
        • <ul id="gckmo"><center id="gckmo"></center></ul>
      •  
  
 
 
 
         
   
         
    
         
    
        恒溫恒濕試驗(yàn)箱
        
	在線(xiàn)咨詢(xún)
    
         
    
         
      淋雨試驗(yàn)箱 
     
        上海林頻儀器股份有限公司Shanghai Linpin Instrument Stock Co Ltd
         
     
        服務(wù)熱線(xiàn)：4000 662 888 
         手機(jī)咨詢(xún)：13818467052
        
    
         
    
         
     
           
	
      
        • 首頁(yè)
          •  
	
	
        • 
	  林頻產(chǎn)品

	  
            
		
          • 試驗(yàn)箱系列
          • 老化箱系列
          • 非標(biāo)定制系列
          • ip防護(hù)系列
          • 振動(dòng)跌落系列
            •  
	  
          

	  
        • 
	  成功案例

	  
            
		 
	  
          

	  
        • 
	  新聞中心

	  
   

## 2.3 Language identification

### Setup: Download fastText language ID model

In [10]:
LID_MODEL_PATH = os.path.join(DATA_DIR, "lid.176.bin")
if not os.path.exists(LID_MODEL_PATH):
    !wget -O {LID_MODEL_PATH} "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
else:
    print(f"Language ID model already exists at {LID_MODEL_PATH}")

Language ID model already exists at /content/LLM-from-scratch/ece405_assignment2/data/lid.176.bin


### (a) Language identification function

In [11]:
from cs336_data.language_identification import identify_language, set_lid_model_path

set_lid_model_path(LID_MODEL_PATH)

# Quick sanity checks
print(identify_language("Hello, this is a test in English."))
print(identify_language("欢迎来到我们的网站"))
print(identify_language("Bonjour, comment allez-vous?"))

('en', 0.9398366808891296)
('zh', 1.000056266784668)
('fr', 0.9740094542503357)


### (c) Run language ID on 20 random WET records, compare to manual judgment

In [12]:
import random
from warcio.archiveiterator import ArchiveIterator

# Collect first 200 WET records, then sample 20
records = []
with open(WET_PATH, "rb") as f:
    for record in ArchiveIterator(f):
        if record.rec_type == "conversion":
            url = record.rec_headers.get_header('WARC-Target-URI')
            content = record.content_stream().read().decode('utf-8', errors='replace')
            if len(content.strip()) > 50:  # skip near-empty records
                records.append((url, content))
            if len(records) >= 200:
                break

random.seed(42)
sample = random.sample(records, 20)

print(f"Sampled {len(sample)} records from {len(records)} total\n")

for i, (url, content) in enumerate(sample):
    lang, score = identify_language(content)
    print(f"Record {i+1:2d} | lang={lang:5s} score={score:.4f} | {url}")
    print(f"           First 120 chars: {content[:120].replace(chr(10), ' ')}")
    print()

Sampled 20 records from 200 total

Record  1 | lang=id    score=0.5478 | http://bagsguccistoer.com/rise%20of%20apollo%20pg
           First 120 chars: แนะนำ 10 rise of apollo pg ไม่ผ่านเอเย่นต์ งบน้อยเล่นได้ ฝากถอนไม่มีขั้นต่ำ INTERNAL FEEDBACK rise of apollo pg สล็อตเว็

Record  2 | lang=zh    score=0.9596 | http://50899.cn/news/16755.html
           First 120 chars: A级毛片无码真人久久久,欧美性饥渴少妇BBB.BBB片 午夜亚洲影院在线观看,日韩欧美精品视频一区二区三区,久免费视频,黄网人妻视频,99视频在线观看精品,午夜A级性爱 BJFQY 經(jīng)營品牌 風(fēng)富圖 佳能 大疆 寶麗

Record  3 | lang=zh    score=0.9951 | http://18sex.v340.info/?&R2=&P=11&OP=&CHANNEL=
           First 120 chars: 本土自拍性愛影片 回 首 頁 │ 點數購買 │ 付款方式 │ 加入會員 │會員登入 │主持人登入 │ 線上客服 │使用說明 │ 使用條款 │ 台妹區 | 內地主播區 | 業績排行 | 會員評價 | 包廂線上人數 | 一對多點數 | 一對一點

Record  4 | lang=en    score=0.8122 | http://beyparkotel.com/
           First 120 chars: Beypark Otel Your browser does not support frames. Untitled Page 

Record  5 | lang=ru    score=0.9911 | http://agrokenya.org/2024/02/10/kak-mozhno-zarabotat-realnye-pin-up-skachat-n

## 2.4 PII masking

### (a)–(c) PII masking functions

In [ ]:
from cs336_data.pii import mask_emails, mask_phone_numbers, mask_ips

# Sanity checks
print(mask_emails("Contact me at test@gmail.com or admin@example.org"))
print(mask_phone_numbers("Call (283)-182-3829 or 2831823829"))
print(mask_ips("Server at 192.0.2.146 and 10.0.0.1"))

### (5) Run PII masking on extracted text, examine 20 random replacements

In [ ]:
import random
from warcio.archiveiterator import ArchiveIterator
from cs336_data.extract import extract_text_from_html_bytes
from cs336_data.pii import mask_emails, mask_phone_numbers, mask_ips

# Extract text from WARC pages and apply PII masking
pii_results = []
with open(WARC_PATH, "rb") as f:
    for record in ArchiveIterator(f):
        if record.rec_type == "response":
            html_bytes = record.content_stream().read()
            text = extract_text_from_html_bytes(html_bytes)
            if text and len(text.strip()) > 100:
                masked_email, n_emails = mask_emails(text)
                masked_phone, n_phones = mask_phone_numbers(masked_email)
                masked_all, n_ips = mask_ips(masked_phone)
                total = n_emails + n_phones + n_ips
                if total > 0:
                    pii_results.append({
                        "url": record.rec_headers.get_header('WARC-Target-URI'),
                        "original": text,
                        "masked": masked_all,
                        "n_emails": n_emails,
                        "n_phones": n_phones,
                        "n_ips": n_ips,
                    })
        if len(pii_results) >= 100:
            break

print(f"Found {len(pii_results)} pages with PII out of first ~100+ WARC responses\n")

# Show 20 random examples
random.seed(42)
sample = random.sample(pii_results, min(20, len(pii_results)))

for i, r in enumerate(sample):
    print(f"{'='*80}")
    print(f"Example {i+1} | URL: {r['url']}")
    print(f"  Emails: {r['n_emails']}, Phones: {r['n_phones']}, IPs: {r['n_ips']}")
    # Show context around each replacement
    for marker in ["|||EMAIL_ADDRESS|||", "|||PHONE_NUMBER|||", "|||IP_ADDRESS|||"]:
        idx = 0
        while True:
            pos = r["masked"].find(marker, idx)
            if pos == -1:
                break
            start = max(0, pos - 40)
            end = min(len(r["masked"]), pos + len(marker) + 40)
            context = r["masked"][start:end].replace("\n", " ")
            print(f"  ...{context}...")
            idx = pos + len(marker)
    print()

## 2.5 Harmful content classification

### Setup: Download Dolma fastText classifiers

In [ ]:
NSFW_MODEL_PATH = os.path.join(DATA_DIR, "dolma_fasttext_nsfw_jigsaw_model.bin")
TOXIC_MODEL_PATH = os.path.join(DATA_DIR, "dolma_fasttext_hatespeech_jigsaw_model.bin")

if not os.path.exists(NSFW_MODEL_PATH):
    !wget -O {NSFW_MODEL_PATH} "https://huggingface.co/allenai/dolma-jigsaw-fasttext-bigrams-nsfw/resolve/main/model.bin"
else:
    print(f"NSFW model already exists at {NSFW_MODEL_PATH}")

if not os.path.exists(TOXIC_MODEL_PATH):
    !wget -O {TOXIC_MODEL_PATH} "https://huggingface.co/allenai/dolma-jigsaw-fasttext-bigrams-hatespeech/resolve/main/model.bin"
else:
    print(f"Toxic speech model already exists at {TOXIC_MODEL_PATH}")

### (1)–(2) NSFW and toxic speech classifiers

In [ ]:
from cs336_data.harmful_content import classify_nsfw, classify_toxic_speech, set_nsfw_model_path, set_toxic_model_path

set_nsfw_model_path(NSFW_MODEL_PATH)
set_toxic_model_path(TOXIC_MODEL_PATH)

# Sanity checks
print("NSFW classifier:")
print(classify_nsfw("This is a normal sentence about cooking."))
print(classify_nsfw("SUCK MY C*CK WIKIPEDIA EDITORS...F*CKING *SSH*LE DORKS."))
print()
print("Toxic speech classifier:")
print(classify_toxic_speech("The weather is nice today."))
print(classify_toxic_speech("What a rude fuck. Arrogant twat who doesn't know what he's talking about."))

### (4) Run on extracted text, compare 20 predictions to own judgment

In [ ]:
import random
from warcio.archiveiterator import ArchiveIterator
from cs336_data.extract import extract_text_from_html_bytes

# Extract text from first 200 WARC pages
texts = []
with open(WARC_PATH, "rb") as f:
    for record in ArchiveIterator(f):
        if record.rec_type == "response":
            html_bytes = record.content_stream().read()
            text = extract_text_from_html_bytes(html_bytes)
            url = record.rec_headers.get_header('WARC-Target-URI')
            if text and len(text.strip()) > 100:
                texts.append((url, text))
        if len(texts) >= 200:
            break

random.seed(42)
sample = random.sample(texts, 20)

print(f"Sampled {len(sample)} pages from {len(texts)} total\n")

for i, (url, text) in enumerate(sample):
    nsfw_label, nsfw_score = classify_nsfw(text)
    toxic_label, toxic_score = classify_toxic_speech(text)
    print(f"Record {i+1:2d} | nsfw={nsfw_label:8s} ({nsfw_score:.4f}) | toxic={toxic_label:9s} ({toxic_score:.4f}) | {url}")
    print(f"           First 120 chars: {text[:120].replace(chr(10), ' ')}")
    print()

## 2.6 Quality Rules (Gopher heuristic filters)

### (a) Gopher quality filter function

In [ ]:
from cs336_data.quality_filters import gopher_quality_filter

# Sanity checks
good_text = ("This should definitely be a valid input text "
             "and of high quality according to Gopher rules. ") * 100
print(f"Good text (1000 words): {gopher_quality_filter(good_text)}")

short_text = "Too short."
print(f"Short text: {gopher_quality_filter(short_text)}")

ellipsis_text = "Line ending with ellipsis...\n" * 100
print(f"Ellipsis text: {gopher_quality_filter(ellipsis_text)}")

numeric_text = "123 456 789 " * 100
print(f"Numeric text: {gopher_quality_filter(numeric_text)}")

### (b) Run Gopher filter on extracted text, compare 20 predictions to own judgment

In [ ]:
import random
from warcio.archiveiterator import ArchiveIterator
from cs336_data.extract import extract_text_from_html_bytes
from cs336_data.quality_filters import gopher_quality_filter
import nltk

# Extract text from first 200 WARC pages
texts = []
with open(WARC_PATH, "rb") as f:
    for record in ArchiveIterator(f):
        if record.rec_type == "response":
            html_bytes = record.content_stream().read()
            text = extract_text_from_html_bytes(html_bytes)
            url = record.rec_headers.get_header('WARC-Target-URI')
            if text and len(text.strip()) > 0:
                texts.append((url, text))
        if len(texts) >= 200:
            break

random.seed(42)
sample = random.sample(texts, 20)

print(f"Sampled {len(sample)} pages from {len(texts)} total\n")

for i, (url, text) in enumerate(sample):
    passes = gopher_quality_filter(text)
    words = nltk.word_tokenize(text)
    alpha_words = [w for w in words if any(c.isalpha() for c in w)]
    n_words = len(alpha_words)
    mean_wl = sum(len(w) for w in alpha_words) / max(n_words, 1)
    lines = text.split('\n')
    ellipsis_pct = sum(1 for l in lines if l.rstrip().endswith('...') or l.rstrip().endswith('\u2026')) / max(len(lines), 1) * 100
    alpha_pct = len(alpha_words) / max(len(words), 1) * 100
    status = 'PASS' if passes else 'FAIL'
    print(f"Record {i+1:2d} | {status} | words={n_words:6d} | mean_wl={mean_wl:.1f} | ellipsis={ellipsis_pct:4.1f}% | alpha={alpha_pct:4.1f}% | {url}")
    print(f"           First 120 chars: {text[:120].replace(chr(10), ' ')}")
    if not passes:
        reasons = []
        if n_words < 50: reasons.append(f'too few words ({n_words})')
        if n_words > 100000: reasons.append(f'too many words ({n_words})')
        if mean_wl < 3: reasons.append(f'mean word too short ({mean_wl:.1f})')
        if mean_wl > 10: reasons.append(f'mean word too long ({mean_wl:.1f})')
        if ellipsis_pct > 30: reasons.append(f'too many ellipsis lines ({ellipsis_pct:.1f}%)')
        if alpha_pct < 80: reasons.append(f'too few alpha words ({alpha_pct:.1f}%)')
        print(f"           Reason: {'; '.join(reasons)}")
    print()

## 2.7 Quality Classifier

### Setup: Download Wikipedia URLs and prepare training data

In [ ]:
# Download Wikipedia reference URLs (43.5M external links from English Wikipedia)
WIKI_URLS_PATH = os.path.join(DATA_DIR, "enwiki-20240420-extracted_urls.txt.gz")
if not os.path.exists(WIKI_URLS_PATH):
    !wget -q -O {WIKI_URLS_PATH} "https://nlp.stanford.edu/data/nfliu/cs336-spring-2024/assignment4/enwiki-20240420-extracted_urls.txt.gz"
else:
    print(f"Wikipedia URLs already exist at {WIKI_URLS_PATH}")

import gzip
with gzip.open(WIKI_URLS_PATH, 'rt') as f:
    all_wiki_urls = [line.strip() for line in f if line.strip()]
print(f"Total Wikipedia reference URLs: {len(all_wiki_urls):,}")

In [ ]:
import random

random.seed(42)
N_POSITIVE = 50000
sampled_urls = random.sample(all_wiki_urls, N_POSITIVE)

# Write sampled URLs to file
SAMPLED_URLS_PATH = os.path.join(DATA_DIR, "sampled_wiki_urls.txt")
with open(SAMPLED_URLS_PATH, 'w') as f:
    for url in sampled_urls:
        f.write(url + '\n')

# Scrape with wget in WARC format
WIKI_WARC_PREFIX = os.path.join(DATA_DIR, "wiki_positive")
WIKI_WARC_PATH = WIKI_WARC_PREFIX + ".warc.gz"

if not os.path.exists(WIKI_WARC_PATH):
    !wget --timeout=5 --tries=1 --no-check-certificate \
        -i {SAMPLED_URLS_PATH} \
        --warc-file={WIKI_WARC_PREFIX} \
        -O /dev/null -q 2>&1 || true

print(f"Wiki WARC: {WIKI_WARC_PATH} ({os.path.getsize(WIKI_WARC_PATH) / 1e6:.1f} MB)")

In [ ]:
import random
from warcio.archiveiterator import ArchiveIterator
from cs336_data.extract import extract_text_from_html_bytes
from cs336_data.quality_filters import gopher_quality_filter

WIKI_WARC_PATH = os.path.join(DATA_DIR, "wiki_positive.warc.gz")

def extract_texts_from_warc(warc_path, max_docs=None, min_length=200):
    """Extract texts from a WARC file."""
    texts = []
    with open(warc_path, 'rb') as f:
        for record in ArchiveIterator(f):
            if record.rec_type == 'response':
                try:
                    html_bytes = record.content_stream().read()
                    text = extract_text_from_html_bytes(html_bytes)
                    if text and len(text.strip()) >= min_length:
                        texts.append(text.strip())
                except Exception:
                    continue
            if max_docs and len(texts) >= max_docs:
                break
    return texts

print("Extracting positive examples (Wikipedia-linked pages)...")
wiki_texts = extract_texts_from_warc(WIKI_WARC_PATH)
print(f"  Raw: {len(wiki_texts)} documents")

print("Extracting negative examples (Common Crawl)...")
cc_texts = extract_texts_from_warc(WARC_PATH, max_docs=10000)
print(f"  Raw: {len(cc_texts)} documents")

print("Applying Gopher quality filters...")
wiki_filtered = [t for t in wiki_texts if gopher_quality_filter(t)]
cc_filtered = [t for t in cc_texts if gopher_quality_filter(t)]
print(f"  After Gopher filter: {len(wiki_filtered)} wiki, {len(cc_filtered)} cc")

n_examples = min(len(wiki_filtered), len(cc_filtered))
if n_examples == 0:
    raise ValueError("No examples after filtering!")
random.seed(42)
wiki_train = random.sample(wiki_filtered, n_examples)
cc_train = random.sample(cc_filtered, n_examples)
print(f"  Balanced: {n_examples} per class ({n_examples * 2} total)")

In [ ]:
import fasttext

TRAIN_PATH = os.path.join(DATA_DIR, "quality_train.txt")
with open(TRAIN_PATH, 'w') as f:
    for text in wiki_train:
        f.write(f"__label__wiki {text.replace(chr(10), ' ').strip()}\n")
    for text in cc_train:
        f.write(f"__label__cc {text.replace(chr(10), ' ').strip()}\n")
print(f"Training file: {n_examples * 2} lines")

QUALITY_MODEL_PATH = os.path.join(DATA_DIR, "quality_classifier.bin")
model = fasttext.train_supervised(
    input=TRAIN_PATH,
    lr=0.1,
    epoch=25,
    wordNgrams=2,
    dim=100,
    loss='softmax',
)
model.save_model(QUALITY_MODEL_PATH)

result = model.test(TRAIN_PATH)
print(f"Labels: {model.labels}")
print(f"Training accuracy: P={result[1]:.4f}, R={result[2]:.4f}")
print(f"Model size: {os.path.getsize(QUALITY_MODEL_PATH) / 1e6:.1f} MB")

### (b) Quality classifier function

In [ ]:
from cs336_data.quality_filters import classify_quality, set_quality_model_path
import pathlib

set_quality_model_path(QUALITY_MODEL_PATH)

fixtures = pathlib.Path(ASSIGNMENT_DIR) / 'tests' / 'fixtures'
with open(fixtures / 'low_quality_cc.txt') as f:
    low_text = f.read()
with open(fixtures / 'high_quality_wiki_reference.txt') as f:
    high_text = f.read()

print("Low quality CC:", classify_quality(low_text))
print("High quality wiki:", classify_quality(high_text))

# Run test
!cd {ASSIGNMENT_DIR} && python -m pytest -xvs -k test_classify_quality tests/ 2>&1 | tail -5

## Section 4: Filter Pipeline & Inspection

### Run filter pipeline on WET file

In [ ]:
import os, sys, pathlib
sys.path.insert(0, ASSIGNMENT_DIR)

# Ensure model paths are set
from cs336_data.language_identification import set_lid_model_path
from cs336_data.harmful_content import set_nsfw_model_path, set_toxic_model_path

DATA_DIR = os.path.join(ASSIGNMENT_DIR, "data")
set_lid_model_path(os.path.join(DATA_DIR, "lid.176.bin"))
set_nsfw_model_path(os.path.join(DATA_DIR, "dolma_fasttext_nsfw_jigsaw_model.bin"))
set_toxic_model_path(os.path.join(DATA_DIR, "dolma_fasttext_hatespeech_jigsaw_model.bin"))

QUALITY_MODEL_PATH = os.path.join(DATA_DIR, "quality_classifier.bin")
if os.path.exists(QUALITY_MODEL_PATH):
    from cs336_data.quality_filters import set_quality_model_path
    set_quality_model_path(QUALITY_MODEL_PATH)

# Run the filter pipeline
FILTERED_DIR = os.path.join(DATA_DIR, "filtered")
WET_PATH = os.path.join(DATA_DIR, "example.warc.wet.gz")
!cd {ASSIGNMENT_DIR} && python scripts/filter_data.py --input_dir {DATA_DIR} --output_dir {FILTERED_DIR} --models_dir {DATA_DIR} --glob "example.warc.wet.gz"

### inspect_filtered_data: Examine kept and discarded examples

In [ ]:
# Inspect kept and discarded examples using the inspection script
!cd {ASSIGNMENT_DIR} && python scripts/inspect_filtered.py \
    --wet_file {WET_PATH} \
    --models_dir {DATA_DIR} \
    --n_kept 5 --n_discarded 5 \
    --output_json {os.path.join(DATA_DIR, "inspection_results.json")}

### tokenize_data: Tokenize filtered data with GPT-2 tokenizer

In [ ]:
!pip install -q transformers

TOKENIZED_PATH = os.path.join(DATA_DIR, "tokenized", "train.bin")
!cd {ASSIGNMENT_DIR} && python scripts/tokenize_data.py --input {FILTERED_DIR} --output {TOKENIZED_PATH}

import numpy as np
tokens = np.fromfile(TOKENIZED_PATH, dtype=np.uint16)
print(f"\nTotal tokens: {len(tokens):,}")
print(f"File size: {os.path.getsize(TOKENIZED_PATH) / 1e6:.1f} MB")

## Section 5: Full-Scale Tokenization (Koa HPC Cluster)

The full-scale tokenization was performed on the Koa HPC cluster (ece405 partition, 16 CPUs, 32 GB RAM) using `scripts/tokenize_data.py` with the `tiktoken` GPT-2 tokenizer.

**Key implementation details:**
- **Streaming architecture**: processes one file at a time, writes tokens incrementally via `struct.pack` as uint16 values — avoids loading all 13 GB of filtered text into memory
- **tiktoken** instead of `transformers.AutoTokenizer` for 10x faster tokenization
- `disallowed_special=()` to handle literal `<|endoftext|>` strings in the filtered data
- GPT-2 end-of-sequence token (ID 50256) appended after each document

**Results:**

| Metric | Value |
|--------|-------|
| Input files | 5,000 filtered text files (13 GB) |
| Documents tokenized | 5,750,675 |
| Total tokens | 8,733,540,502 (~8.7B) |
| Output file | `train.bin` (17 GB, uint16 numpy) |
| Processing time | 24.6 minutes (16 workers) |

The validation split (`valid.bin`, 10M tokens) was carved from the end of `train.bin` using `scripts/split_validation.py`.

Slurm job: `scripts/tokenize_job.slurm`

## Section 6: GPT-2 Training (train_model)

### Configuration

Training was performed on the Koa HPC cluster using the staff-provided `cs336-basics/scripts/train.py` with DDP (DistributedDataParallel).

| Parameter | Value |
|-----------|-------|
| Model | GPT-2 small (124M params), 12 layers, 768 hidden dim, 12 heads |
| Hardware | 2x NVIDIA RTX A4000 (16 GB each), Koa ece405 partition |
| Batch size per device | 16 |
| Gradient accumulation steps | 8 |
| Effective batch size | 16 x 8 x 2 GPUs = 256 sequences/step |
| Tokens per step | 256 x 512 = 131,072 |
| Precision | bfloat16 |
| torch.compile | Disabled (OOM on A4000) |
| Training steps | 12,000 |
| Eval interval | Every 1,000 steps |
| Total tokens seen | ~1.57B (18% of 8.7B dataset) |
| Training time | ~14 hours |
| wandb | offline mode, synced post-hoc |

**Hardware adaptation note**: The default training configuration (batch_size=128, torch.compile=True, 200K steps) was designed for the Together cluster's A100 GPUs (40-80 GB VRAM). On Koa's RTX A4000 GPUs (16 GB), we reduced batch size from 128 to 16 and disabled torch.compile to fit in VRAM, compensating with gradient accumulation (8 steps) to preserve the same effective batch size of 256 sequences per step. Training steps were reduced from 200K to 12,000 to fit within the 20-hour Slurm wall time limit.

Config: `cs336-basics/configs/experiment/your_data.yaml` | Slurm job: `scripts/train_job.slurm`

### Validation loss curve

Validation was measured on a 10M-token held-out split from our own filtered corpus (Paloma C4 100 domains benchmark was not available on the Koa cluster).

| Step | Val Loss | Wall Time |
|------|----------|-----------|
| 1,000 | 4.032 | 1h 09m |
| 2,000 | 3.683 | 2h 12m |
| 3,000 | 3.507 | 3h 21m |
| 4,000 | 3.379 | 4h 34m |
| 5,000 | 3.273 | 5h 47m |
| 6,000 | 3.202 | 6h 58m |
| 7,000 | 3.101 | 8h 08m |
| 8,000 | 3.030 | 9h 17m |
| 9,000 | 2.982 | 10h 25m |
| 10,000 | 2.927 | 11h 33m |
| 11,000 | 2.883 | 12h 42m |
| **12,000** | **2.856** | **13h 55m** |

**Best validation loss: 2.856** at step 12,000 (final step).

The loss decreased consistently throughout training with no signs of plateauing, suggesting further training would improve results.

wandb run: https://wandb.ai/pavelbushuyeu-university-of-hawaii-system/ece405-data/runs/6n6ms27f